# WarpAUV Trajectory Tracking Experiments

This notebook is the control panel for trajectory-tracking experiments. It can launch IsaacLab training, evaluate checkpoints, and visualize every evaluated checkpoint found under the selected run directory.

Default cells are safe: training/eval commands are previewed unless `RUN_TRAIN` or `RUN_EVAL` is set to `True`.


In [ ]:
from pathlib import Path

# IsaacLab workspace. Commands below run from this directory.
ISAACLAB_ROOT = Path("/home/jining_yang/IsaacLab")

TASK_NAME = "Isaac-WarpAUV-Traj-Direct-v1"
EXPERIMENT_NAME = "warpauv_traj_heavy2_direct"

# Leave LOAD_RUN empty to use the newest run under logs/rsl_rl/EXPERIMENT_NAME.
LOAD_RUN = ""

# Training controls.
RUN_TRAIN = True
TRAIN_NUM_ENVS = 2048
TRAIN_HEADLESS = True
TRAIN_EXTRA_ARGS = []  # e.g. ["--logger", "tensorboard", "--run_name", "curriculum_v1"]
AUTO_SELECT_LATEST_AFTER_TRAIN = True

# Evaluation controls.
RUN_EVAL = False
EVAL_CHECKPOINTS = "latest"  # evaluate only the final saved policy by default
EVAL_TRAJECTORIES = ["lissajous", "helix", "spiral"]
EVAL_DURATION = 32.0
EVAL_HEADLESS = True
EVAL_SKIP_EXISTING = True
ALIGN_INITIAL_TARGET = True
RANDOM_CURVE_COUNT = 8

# Optional trajectory overrides for eval. Use None to keep env defaults.
TRAJECTORY_AMP_X = None
TRAJECTORY_AMP_Y = None
TRAJECTORY_AMP_Z = None
TRAJECTORY_PERIOD = None
TRAJECTORY_RADIUS_MIN = None
TRAJECTORY_RADIUS_MAX = None

# Plot controls.
DETAIL_CHECKPOINT = "latest"  # "latest", "best", or explicit checkpoint name
DETAIL_TRAJECTORY = "lissajous"
DETAIL_ENV_ID = 0
SAVE_FIGURES = True

TRAIN_SCRIPT = "scripts/reinforcement_learning/rsl_rl/train.py"
EVAL_SCRIPT = "source/isaaclab_tasks/isaaclab_tasks/direct/isaac-auv-env/custom_workflows/play_trajectory_eval.py"
TRAJECTORY_NAMES = ["lissajous", "helix", "spiral", "chirp", "racetrack", "random_smooth"]


## Helpers


In [ ]:
import math
import os
import re
import shlex
import subprocess
import time
from collections.abc import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


def shell_join(command):
    return " ".join(shlex.quote(str(part)) for part in command)


def run_command(command, *, cwd=ISAACLAB_ROOT, execute=False, label=None, extra_env=None):
    command = [str(part) for part in command]
    if label:
        print(f"[{label}]")
    print(shell_join(command))
    if not execute:
        return None

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
    if extra_env:
        env.update(extra_env)

    start = time.time()
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    elapsed = time.time() - start
    print(f"\n[exit={return_code}] elapsed={elapsed/60:.1f} min")
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {shell_join(command)}")
    return return_code


def logs_root():
    return ISAACLAB_ROOT / "logs" / "rsl_rl" / EXPERIMENT_NAME


def results_root(run_name=None):
    return ISAACLAB_ROOT / "source" / "results" / "rsl_rl" / EXPERIMENT_NAME / resolve_run(run_name)


def checkpoint_iter(name):
    match = re.search(r"model_(\d+)\.pt$", str(name))
    return int(match.group(1)) if match else -1


def list_runs():
    root = logs_root()
    if not root.exists():
        return []
    return sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)


def latest_run_dir(required=True):
    runs = list_runs()
    if not runs:
        if not required:
            return None
        raise FileNotFoundError(f"No runs found under {logs_root()}")
    return runs[0]


def resolve_run(run_name=None, required=True):
    run_name = LOAD_RUN if run_name is None else run_name
    if run_name:
        return run_name
    latest = latest_run_dir(required=required)
    return latest.name if latest is not None else None


def checkpoints_for_run(run_name=None):
    run = resolve_run(run_name)
    run_dir = logs_root() / run
    if not run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_dir}")
    return sorted([p.name for p in run_dir.glob("model_*.pt")], key=checkpoint_iter)


def resolve_checkpoints(selection, run_name=None):
    available = checkpoints_for_run(run_name)
    if isinstance(selection, str):
        if selection == "all":
            return available
        if selection == "latest":
            if not available:
                raise FileNotFoundError(f"No checkpoints found in run {resolve_run(run_name)}")
            return [available[-1]]
        return [selection]
    if isinstance(selection, Iterable):
        resolved = []
        for item in selection:
            resolved.extend(resolve_checkpoints(item, run_name))
        return list(dict.fromkeys(resolved))
    raise TypeError(f"Unsupported checkpoint selection: {selection!r}")


def eval_dir_name(checkpoint, trajectory):
    stem = Path(checkpoint).stem
    if trajectory == "lissajous":
        return f"{stem}_trajectory_eval"
    return f"{stem}_{trajectory}_trajectory_eval"


def eval_dir(checkpoint, trajectory, run_name=None):
    return results_root(run_name) / eval_dir_name(checkpoint, trajectory)


def summary_path(checkpoint, trajectory, run_name=None):
    return eval_dir(checkpoint, trajectory, run_name) / "summary_metrics.csv"


def logs_path(checkpoint, trajectory, run_name=None):
    return eval_dir(checkpoint, trajectory, run_name) / "logs.csv"


def checkpoint_exists(checkpoint, run_name=None):
    return (logs_root() / resolve_run(run_name) / checkpoint).exists()


def show_active_paths():
    print(f"IsaacLab root: {ISAACLAB_ROOT}")
    print(f"Logs root:     {logs_root()}")
    run = resolve_run(required=False)
    if run is None:
        print("Active run:    <none yet>")
        print("Run the training cell first to create the first heavy2 trajectory policy.")
        return
    print(f"Active run:    {run}")
    print(f"Run logs:      {logs_root() / run}")
    print(f"Run results:   {results_root(run)}")

show_active_paths()


## Runs And Checkpoints

This cell discovers run folders and checkpoints directly from `logs/rsl_rl/warpauv_traj_heavy2_direct`.


In [ ]:
run_rows = []
for run_dir in list_runs():
    ckpts = sorted([p.name for p in run_dir.glob("model_*.pt")], key=checkpoint_iter)
    run_rows.append({
        "run": run_dir.name,
        "modified": pd.to_datetime(run_dir.stat().st_mtime, unit="s"),
        "num_checkpoints": len(ckpts),
        "latest_checkpoint": ckpts[-1] if ckpts else "",
    })

runs_df = pd.DataFrame(run_rows)
display(runs_df.head(20))

active_run = resolve_run(required=False)
if active_run is None:
    active_checkpoints = []
    print(f"No runs found under {logs_root()}. Run training first.")
else:
    active_checkpoints = checkpoints_for_run(active_run)
    print(f"Active run: {active_run}")
    print(f"Checkpoints ({len(active_checkpoints)}): {active_checkpoints}")


## Train From Notebook

Set `RUN_TRAIN=True` in the config cell and run this cell to launch RSL-RL training. With the current repo config this uses the trajectory curriculum and PPO settings in `agents/rsl_rl_ppo_cfg.py`.


In [ ]:
def build_train_command():
    command = [
        "./isaaclab.sh",
        "-p",
        TRAIN_SCRIPT,
        "--task",
        TASK_NAME,
        "--num_envs",
        str(TRAIN_NUM_ENVS),
    ]
    if TRAIN_HEADLESS:
        command.append("--headless")
    command.extend(TRAIN_EXTRA_ARGS)
    return command


def train_policy(execute=RUN_TRAIN):
    before = {p.name for p in list_runs()}
    result = run_command(build_train_command(), execute=execute, label="TRAIN")
    if execute and AUTO_SELECT_LATEST_AFTER_TRAIN:
        after = list_runs()
        if after:
            print(f"Newest run after training: {after[0].name}")
    return result

train_policy(execute=RUN_TRAIN)
if not RUN_TRAIN:
    print("RUN_TRAIN is False; training command was previewed only.")


## Evaluate Checkpoints

Set `RUN_EVAL=True` to evaluate the final saved policy for the selected run. Existing `summary_metrics.csv` files are skipped when `EVAL_SKIP_EXISTING=True`.


In [ ]:
def validate_trajectories(trajectories):
    if isinstance(trajectories, str):
        trajectories = [trajectories]
    unknown = [name for name in trajectories if name not in TRAJECTORY_NAMES]
    if unknown:
        raise ValueError(f"Unknown trajectories: {unknown}. Choices: {TRAJECTORY_NAMES}")
    return list(trajectories)


def build_eval_command(checkpoint, trajectory, run_name=None):
    run = resolve_run(run_name)
    command = [
        "./isaaclab.sh",
        "-p",
        EVAL_SCRIPT,
        "--task",
        TASK_NAME,
        "--load_run",
        run,
        "--checkpoint",
        checkpoint,
        "--trajectory",
        trajectory,
        "--duration",
        str(EVAL_DURATION),
    ]
    if EVAL_HEADLESS:
        command.append("--headless")
    if ALIGN_INITIAL_TARGET:
        command.append("--align_initial_target")
    if trajectory == "random_smooth":
        command.extend(["--random_curve_count", str(RANDOM_CURVE_COUNT)])
    optional_pairs = [
        ("--trajectory_amp_x", TRAJECTORY_AMP_X),
        ("--trajectory_amp_y", TRAJECTORY_AMP_Y),
        ("--trajectory_amp_z", TRAJECTORY_AMP_Z),
        ("--trajectory_period", TRAJECTORY_PERIOD),
        ("--trajectory_radius_min", TRAJECTORY_RADIUS_MIN),
        ("--trajectory_radius_max", TRAJECTORY_RADIUS_MAX),
    ]
    for flag, value in optional_pairs:
        if value is not None:
            command.extend([flag, str(value)])
    return command


def run_eval_matrix(checkpoints=None, trajectories=None, run_name=None, execute=RUN_EVAL, skip_existing=EVAL_SKIP_EXISTING):
    try:
        run = resolve_run(run_name)
    except FileNotFoundError as exc:
        if execute:
            raise FileNotFoundError(f"{exc}. Train a policy before running eval.") from exc
        print(f"RUN_EVAL is False and no run exists yet: {exc}")
        return []
    checkpoints = resolve_checkpoints(EVAL_CHECKPOINTS if checkpoints is None else checkpoints, run)
    trajectories = validate_trajectories(EVAL_TRAJECTORIES if trajectories is None else trajectories)
    commands = []
    for checkpoint in checkpoints:
        if not checkpoint_exists(checkpoint, run):
            raise FileNotFoundError(f"Checkpoint not found: {logs_root() / run / checkpoint}")
        for trajectory in trajectories:
            out_summary = summary_path(checkpoint, trajectory, run)
            if skip_existing and out_summary.exists():
                print(f"[SKIP] {checkpoint} / {trajectory}: {out_summary}")
                continue
            command = build_eval_command(checkpoint, trajectory, run)
            commands.append(command)
            run_command(command, execute=execute, label=f"EVAL {checkpoint} / {trajectory}")
    return commands

planned_eval_commands = run_eval_matrix(execute=RUN_EVAL)
if not RUN_EVAL:
    print(f"RUN_EVAL is False; {len(planned_eval_commands)} eval commands were previewed only.")


## Auto-Collect All Checkpoint Metrics

Visualization starts here. These cells scan every `summary_metrics.csv` under the selected results directory and automatically include all evaluated checkpoints.


In [ ]:
def parse_eval_dir(path):
    name = Path(path).parent.name if Path(path).name == "summary_metrics.csv" else Path(path).name
    match = re.match(r"model_(\d+)(?:_(.*))?_trajectory_eval$", name)
    if not match:
        return None, None
    iteration = int(match.group(1))
    suffix = match.group(2)
    trajectory = "lissajous" if suffix is None else suffix
    return iteration, trajectory


def collect_summary_df(run_name=None):
    run = resolve_run(run_name, required=False)
    if run is None:
        return pd.DataFrame(columns=["run", "checkpoint", "checkpoint_name", "trajectory"])
    root = results_root(run)
    rows = []
    for csv_path in sorted(root.glob("model_*_trajectory_eval/summary_metrics.csv")):
        iteration, inferred_trajectory = parse_eval_dir(csv_path)
        if iteration is None:
            continue
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        row = df.iloc[0].to_dict()
        trajectory = row.get("trajectory", inferred_trajectory)
        row.update({
            "run": run,
            "checkpoint": iteration,
            "checkpoint_name": f"model_{iteration}.pt",
            "trajectory": trajectory,
            "summary_path": str(csv_path),
            "logs_path": str(csv_path.parent / "logs.csv"),
        })
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["run", "checkpoint", "checkpoint_name", "trajectory"])
    summary = pd.DataFrame(rows)
    numeric_cols = ["position_rmse", "position_mae", "max_position_error", "velocity_rmse", "num_curves"]
    for col in numeric_cols:
        if col in summary.columns:
            summary[col] = pd.to_numeric(summary[col], errors="coerce")
    summary = summary.sort_values(["trajectory", "checkpoint"]).reset_index(drop=True)
    return summary


def save_summary_table(summary_df, run_name=None):
    out = results_root(run_name) / "checkpoint_rmse_summary.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(out, index=False)
    print(f"Saved: {out}")
    return out

summary_df = collect_summary_df()
if summary_df.empty:
    run = resolve_run(required=False)
    if run is None:
        print(f"No runs found under: {logs_root()}")
    else:
        print(f"No evaluated summaries found under: {results_root(run)}")
else:
    save_summary_table(summary_df)
    display(summary_df[["trajectory", "checkpoint_name", "position_rmse", "position_mae", "max_position_error", "velocity_rmse"]])


## RMSE Curves For All Evaluated Checkpoints


In [ ]:
def plot_rmse_summary(summary_df, run_name=None, save=SAVE_FIGURES):
    if summary_df.empty:
        raise ValueError("summary_df is empty. Run eval first or select a run with results.")
    fig, ax = plt.subplots(figsize=(11, 5.8), constrained_layout=True)
    for trajectory, group in summary_df.groupby("trajectory"):
        group = group.sort_values("checkpoint")
        ax.plot(group["checkpoint"], group["position_rmse"], marker="o", linewidth=2, label=trajectory)
    ax.set_title(f"Trajectory Tracking Position RMSE: {resolve_run(run_name)}")
    ax.set_xlabel("checkpoint iteration")
    ax.set_ylabel("position RMSE [m]")
    ax.legend(ncol=2)
    out = results_root(run_name) / "checkpoint_rmse_curve.png"
    if save:
        fig.savefig(out, dpi=180)
        print(f"Saved: {out}")
    return fig, ax

if not summary_df.empty:
    fig, ax = plot_rmse_summary(summary_df)
    plt.show()


## Checkpoint x Trajectory Heatmap


In [ ]:
def plot_rmse_heatmap(summary_df, run_name=None, save=SAVE_FIGURES):
    if summary_df.empty:
        raise ValueError("summary_df is empty.")
    pivot = summary_df.pivot_table(index="checkpoint_name", columns="trajectory", values="position_rmse", aggfunc="mean")
    pivot = pivot.reindex(sorted(pivot.index, key=checkpoint_iter))
    fig, ax = plt.subplots(figsize=(max(8, 1.15 * len(pivot.columns)), max(4, 0.36 * len(pivot))), constrained_layout=True)
    image = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Position RMSE heatmap [m]")
    fig.colorbar(image, ax=ax, label="position RMSE [m]")
    out = results_root(run_name) / "checkpoint_rmse_heatmap.png"
    if save:
        fig.savefig(out, dpi=180)
        print(f"Saved: {out}")
    return fig, ax, pivot

if not summary_df.empty:
    fig, ax, rmse_pivot = plot_rmse_heatmap(summary_df)
    plt.show()
    display(rmse_pivot)


## Best Checkpoints By Trajectory


In [ ]:
def best_by_trajectory(summary_df):
    if summary_df.empty:
        return pd.DataFrame()
    idx = summary_df.groupby("trajectory")["position_rmse"].idxmin()
    cols = ["trajectory", "checkpoint_name", "position_rmse", "position_mae", "max_position_error", "velocity_rmse"]
    return summary_df.loc[idx, cols].sort_values("trajectory").reset_index(drop=True)

best_df = best_by_trajectory(summary_df)
display(best_df)


## Load And Plot One Evaluation Log

`DETAIL_CHECKPOINT="latest"` selects the largest evaluated checkpoint for `DETAIL_TRAJECTORY`. `DETAIL_CHECKPOINT="best"` selects the lowest RMSE checkpoint for that trajectory.


In [ ]:
def evaluated_checkpoints_for_trajectory(summary_df, trajectory):
    if summary_df.empty:
        return []
    sub = summary_df[summary_df["trajectory"] == trajectory].sort_values("checkpoint")
    return sub["checkpoint_name"].tolist()


def resolve_detail_trajectory(summary_df, preferred=DETAIL_TRAJECTORY):
    if summary_df.empty:
        raise ValueError("summary_df is empty.")
    available = sorted(summary_df["trajectory"].unique().tolist())
    if preferred in available:
        return preferred
    fallback = available[0]
    print(f"Trajectory {preferred!r} has no evaluated logs in this run; using {fallback!r} instead.")
    return fallback


def resolve_detail_checkpoint(summary_df, trajectory, selection=DETAIL_CHECKPOINT):
    sub = summary_df[summary_df["trajectory"] == trajectory].sort_values("checkpoint")
    if sub.empty:
        raise FileNotFoundError(f"No evaluated summaries for trajectory={trajectory!r}")
    if selection == "latest":
        return sub.iloc[-1]["checkpoint_name"]
    if selection == "best":
        return sub.loc[sub["position_rmse"].idxmin()]["checkpoint_name"]
    return selection


def load_eval_log(checkpoint, trajectory, run_name=None):
    path = logs_path(checkpoint, trajectory, run_name)
    if not path.exists():
        raise FileNotFoundError(f"Missing logs.csv: {path}")
    df = pd.read_csv(path)
    if "env_id" not in df.columns:
        df["env_id"] = 0
    return df, path


def plot_eval_detail(log_df, checkpoint, trajectory, env_id=0, run_name=None, save=SAVE_FIGURES):
    df = log_df[log_df["env_id"] == env_id].copy() if "env_id" in log_df.columns else log_df.copy()
    if df.empty:
        raise ValueError(f"No rows for env_id={env_id}")
    pos_rmse = float(np.sqrt(np.mean(df["position_error"] ** 2)))
    vel_rmse = float(np.sqrt(np.mean(df["velocity_error"] ** 2)))

    fig = plt.figure(figsize=(13, 9), constrained_layout=True)
    gs = fig.add_gridspec(3, 2)

    ax0 = fig.add_subplot(gs[0:2, 0])
    ax0.plot(df["desired_x"], df["desired_y"], label="desired", linewidth=2.2)
    ax0.plot(df["true_x"], df["true_y"], label="actual", linewidth=1.7)
    ax0.set_title(f"XY path: {checkpoint} / {trajectory} / env {env_id}")
    ax0.set_xlabel("x [m]")
    ax0.set_ylabel("y [m]")
    ax0.axis("equal")
    ax0.legend()

    ax1 = fig.add_subplot(gs[0, 1])
    for axis in ["x", "y", "z"]:
        ax1.plot(df["time"], df[f"desired_{axis}"], linestyle="--", alpha=0.75, label=f"{axis} desired")
        ax1.plot(df["time"], df[f"true_{axis}"], label=f"{axis} actual")
    ax1.set_title("Position tracking")
    ax1.set_xlabel("time [s]")
    ax1.set_ylabel("position [m]")
    ax1.legend(ncol=2, fontsize=8)

    ax2 = fig.add_subplot(gs[1, 1])
    ax2.plot(df["time"], df["position_error"], label=f"position error; RMSE {pos_rmse:.3f} m")
    ax2.set_title("Position error")
    ax2.set_xlabel("time [s]")
    ax2.set_ylabel("error [m]")
    ax2.legend()

    ax3 = fig.add_subplot(gs[2, 0])
    for axis in ["x", "y", "z"]:
        ax3.plot(df["time"], df[f"desired_v{axis}"], linestyle="--", alpha=0.75, label=f"v{axis} desired")
        ax3.plot(df["time"], df[f"true_v{axis}"], label=f"v{axis} actual")
    ax3.set_title("Velocity tracking")
    ax3.set_xlabel("time [s]")
    ax3.set_ylabel("velocity [m/s]")
    ax3.legend(ncol=2, fontsize=8)

    ax4 = fig.add_subplot(gs[2, 1])
    ax4.plot(df["time"], df["velocity_error"], label=f"velocity error; RMSE {vel_rmse:.3f} m/s")
    ax4.plot(df["time"], df["action_norm"], alpha=0.75, label="action norm")
    ax4.set_title("Velocity error and action norm")
    ax4.set_xlabel("time [s]")
    ax4.legend()

    out = eval_dir(checkpoint, trajectory, run_name) / f"tracking_eval_env{env_id}.png"
    if save:
        fig.savefig(out, dpi=180)
        print(f"Saved: {out}")
    return fig

if not summary_df.empty:
    detail_trajectory = resolve_detail_trajectory(summary_df, DETAIL_TRAJECTORY)
    detail_checkpoint = resolve_detail_checkpoint(summary_df, detail_trajectory, DETAIL_CHECKPOINT)
    detail_df, detail_log_path = load_eval_log(detail_checkpoint, detail_trajectory)
    print(f"Loaded {len(detail_df)} rows from {detail_log_path}")
    fig = plot_eval_detail(detail_df, detail_checkpoint, detail_trajectory, DETAIL_ENV_ID)
    plt.show()
    display(detail_df.head())


## Gallery For One Checkpoint

This automatically reads every trajectory log available for the selected checkpoint.


In [ ]:
def available_trajectories_for_checkpoint(summary_df, checkpoint):
    if summary_df.empty:
        return []
    sub = summary_df[summary_df["checkpoint_name"] == checkpoint]
    return sorted(sub["trajectory"].unique().tolist())


def plot_checkpoint_gallery(summary_df, checkpoint=None, run_name=None, save=SAVE_FIGURES):
    if summary_df.empty:
        raise ValueError("summary_df is empty.")
    if checkpoint is None or checkpoint in {"latest", "best"}:
        trajectory = resolve_detail_trajectory(summary_df, DETAIL_TRAJECTORY)
        checkpoint = resolve_detail_checkpoint(summary_df, trajectory, checkpoint or "latest")
    trajectories = available_trajectories_for_checkpoint(summary_df, checkpoint)
    if not trajectories:
        raise FileNotFoundError(f"No evaluated trajectories for {checkpoint}")

    fig, axes = plt.subplots(len(trajectories), 2, figsize=(13, 3.2 * len(trajectories)), constrained_layout=True)
    if len(trajectories) == 1:
        axes = np.array([axes])

    for row, trajectory in enumerate(trajectories):
        df, path = load_eval_log(checkpoint, trajectory, run_name)
        if "env_id" in df.columns:
            df = df[df["env_id"] == df["env_id"].min()].copy()
        ax_xy, ax_err = axes[row]
        ax_xy.plot(df["desired_x"], df["desired_y"], label="desired", linewidth=2)
        ax_xy.plot(df["true_x"], df["true_y"], label="actual", linewidth=1.5)
        ax_xy.set_title(f"{trajectory}: XY")
        ax_xy.set_xlabel("x [m]")
        ax_xy.set_ylabel("y [m]")
        ax_xy.axis("equal")
        ax_xy.legend()

        rmse = float(np.sqrt(np.mean(df["position_error"] ** 2)))
        ax_err.plot(df["time"], df["position_error"], label=f"position RMSE {rmse:.3f} m")
        ax_err.plot(df["time"], df["velocity_error"], alpha=0.75, label="velocity error")
        ax_err.set_title(f"{trajectory}: errors")
        ax_err.set_xlabel("time [s]")
        ax_err.legend()

    out = results_root(run_name) / f"{Path(checkpoint).stem}_trajectory_gallery.png"
    if save:
        fig.savefig(out, dpi=180)
        print(f"Saved: {out}")
    return fig

if not summary_df.empty:
    gallery_trajectory = resolve_detail_trajectory(summary_df, DETAIL_TRAJECTORY)
    gallery_checkpoint = resolve_detail_checkpoint(summary_df, gallery_trajectory, DETAIL_CHECKPOINT)
    fig = plot_checkpoint_gallery(summary_df, gallery_checkpoint)
    plt.show()


## Quick Numeric Report


In [ ]:
if not summary_df.empty:
    numeric_report = summary_df.groupby("trajectory").agg(
        evaluated_checkpoints=("checkpoint_name", "nunique"),
        best_position_rmse=("position_rmse", "min"),
        median_position_rmse=("position_rmse", "median"),
        best_velocity_rmse=("velocity_rmse", "min"),
    ).reset_index()
    display(numeric_report)

    if 'detail_df' in globals():
        env_df = detail_df[detail_df.get("env_id", 0) == DETAIL_ENV_ID] if "env_id" in detail_df.columns else detail_df
        detail_metrics = pd.Series({
            "position_rmse_m": float(np.sqrt(np.mean(env_df["position_error"] ** 2))),
            "position_mae_m": float(env_df["position_error"].mean()),
            "position_p90_m": float(env_df["position_error"].quantile(0.90)),
            "position_max_m": float(env_df["position_error"].max()),
            "velocity_rmse_mps": float(np.sqrt(np.mean(env_df["velocity_error"] ** 2))),
            "velocity_mae_mps": float(env_df["velocity_error"].mean()),
            "attitude_mae_rad": float(env_df["attitude_error"].mean()),
            "action_norm_mean": float(env_df["action_norm"].mean()),
            "action_norm_p95": float(env_df["action_norm"].quantile(0.95)),
        })
        display(detail_metrics)
